<img src="https://www.th-koeln.de/img/logo.svg" style="float:right;" width="200">

# 12th exercise: <font color="#C70039">First reinforcement-learning game (*Frozen Lake*) using Gymnasium</font>
* Course: AML
* Lecturer: <a href="https://www.gernotheisenberg.de/">Gernot Heisenberg</a>
* Author of notebook: <a href="https://www.gernotheisenberg.de/">Gernot Heisenberg</a>. This notebook is based on the post and notebook by [Rodolfo Mendes](https://morioh.com/p/18a96b9091d3).
* Date: 01.08.2026

<img src="https://miro.medium.com/v2/resize:fit:720/format:webp/1*i53DAlKJx_91HgcSiFwyJQ.png" style="float: center;" width="600">

---------------------------------
**GENERAL NOTE 1**:  
Please read the entire notebook. Its markdown cells and code comments explain how the individual steps work together.

**GENERAL NOTE 2**:  
* Use English for source-code comments and written observations.
* This applies to all exercises throughout this course.

---------------------------------

### <font color="ce33ff">DESCRIPTION</font>:

#### Gymnasium
Gymnasium is the maintained successor to OpenAI Gym. It provides a common API for reinforcement-learning environments. Install the environment used here with `pip install "gymnasium[toy-text]"`.

#### Frozen Lake
The agent must cross a slippery frozen lake. `S` is the start, `F` is safe frozen surface, `H` is a hole, and `G` is the goal. The episode ends when the agent reaches the goal or falls into a hole. It receives reward `1` at the goal and `0` otherwise.

* SFFF
* FHFH
* FFFH
* HFFG

<img src="./images/FrozenLake.States.Rewards.png" style="float: center;" width="800">

---------------------------------

### <font color="FFC300">TASKS</font>:
1. Import the notebook into Google Colab or use your local machine.
2. Add your name, matriculation number, and date below the notebook author.
3. Read and run the entire notebook, and explain the important steps.
4. Install Gymnasium with the toy-text environments.
5. Train the Q-learning agent and examine how its performance changes.
6. Create a test plan for the hyperparameters and document your observations.
---------------------------------

### Imports 
import all important libs including gym

In [ ]:
# Google Colab setup: make repository files available under the expected relative paths.
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    repository = "/content/AML"
    if not os.path.isdir(repository):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/gheisenberg/AML.git", repository], check=True)
    os.chdir(repository)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "gymnasium[toy-text]"], check=True)

In [1]:
# Gymnasium supplies the environment and NumPy stores the Q-table.
# The notebook uses Gymnasium's current reset and step API.
import gymnasium as gym
import numpy as np

ModuleNotFoundError: No module named 'gymnasium'

In [ ]:
# Print the installed version to make results and API behaviour easier to reproduce.
# Print the installed version to make results and API behaviour easier to reproduce.
print(gym.__version__)

### Create the environment
`gym.make` constructs the environment. The current API returns `(observation, info)` from `reset` and `(observation, reward, terminated, truncated, info)` from `step`.

In [ ]:
# Slippery ice makes action outcomes stochastic, which creates the learning challenge.
# Seeds make exploration and environment behaviour reproducible.
env = gym.make("FrozenLake-v1", is_slippery=True)
env.action_space.seed(1)
rng = np.random.default_rng(1)

### Creating the Q-Table
Now, construct your Q-table, and initialize all the Q-values to zero for each state-action pair.
The number of rows in the table is equivalent to the size of the state space in the environment, and the number of columns is equivalent to the size of the action space (see above). You can get this information using *env.observation_space.n* and *env.action_space.n* as shown below in the code. Then, you can use this information to build the Q-table and initialize it with zeros.

In [ ]:
# Create one Q-value for every combination of state and action.
# Values start at zero because the agent has no experience yet.
action_space_size = env.action_space.n
state_space_size = env.observation_space.n

q_table = np.zeros((state_space_size, action_space_size))

In [ ]:
print(q_table)

### Initializing Q-Learning hyperparameters
Now, we're going to create and initialize all the parameters needed to implement the Q-learning algorithm.

First, with *num_episodes*, you define the total number of episodes you want the agent to play during training. Then, with *max_steps_per_episode*, you define a maximum number of steps that your agent is allowed to take within a single episode. So, if by the 100th step, the agent has not reached the frisbee or fallen through a hole, then the episode will terminate with the agent receiving zero points.

Next, you will set your *learning_rate* and your *discount_rate* as well, which was represented with the symbol (lambda) in the course slides (keyword: discounted return G_t).

Now, the last four parameters are all related to the exploration-exploitation dilemma with respect to the epsilon-greedy policy. You are initializing your *exploration_rate* to **1** and setting the *max_exploration_rate* to **1** and a *min_exploration_rate* to **0.01**. The *max* and *min* are just bounds to how large or small your exploration rate can be. Remember, the exploration rate was represented with the symbol (epsilon) when discussed in the course slides.

Lastly, you will set the *exploration_decay_rate* to **0.01** to determine the rate at which the *exploration_rate* will decay.

**YOUR <font color="FFC300">TASK</font> in this exercise is as follows** (point 7 from the task list above):

All of the above parameters can change!
Your task is to create a *testplan* and tune all parameters by yourself and observe how they influence and change the performance of the algorithm. 
Make notes! They will help you during the exam.

In [ ]:
# Exploration begins high so the agent first discovers useful transitions.
# The decay schedule gradually shifts behaviour toward the best known actions.
num_episodes = 10_000
max_steps_per_episode = 200

learning_rate = 0.1
discount_rate = 0.99

exploration_rate = 1.0
max_exploration_rate = 1.0
min_exploration_rate = 0.01
exploration_decay_rate = 0.001

Create a list to hold all of the rewards you will get from each episode. 
By means of this you can observe how your game score changes over time.

In [ ]:
# Store one total reward per episode to measure learning progress later.
# Store one total reward per episode to measure learning progress later.
rewards_all_episodes = []

In the following code section, the entire Q-learning algorithm is implemented as discussed in detail in the AML course. 
When this code is executed, this is exactly where the training will take place. 
* The first for-loop contains everything that happens within a single episode. 
* The second nested loop contains everything that happens for a single time-step.

Read all the red comments, as they contain lots of important information on the implementation.

In [ ]:
# Each episode alternates between action selection, environment feedback, and a Q-value update.
# Terminal states do not bootstrap from a future value because no future step follows.
# Q-learning algorithm
for episode in range(num_episodes):
    state, info = env.reset(seed=episode)
    episode_reward = 0.0

    for _ in range(max_steps_per_episode):
        # Follow the greedy policy or explore with a random action.
        if rng.random() > exploration_rate:
            action = int(np.argmax(q_table[state]))
        else:
            action = env.action_space.sample()

        new_state, reward, terminated, truncated, info = env.step(action)
        episode_finished = terminated or truncated

        # Terminal states have no discounted future return.
        future_value = 0.0 if episode_finished else np.max(q_table[new_state])
        temporal_difference = reward + discount_rate * future_value - q_table[state, action]
        q_table[state, action] += learning_rate * temporal_difference

        state = new_state
        episode_reward += reward
        if episode_finished:
            break

    exploration_rate = min_exploration_rate + (
        max_exploration_rate - min_exploration_rate
    ) * np.exp(-exploration_decay_rate * episode)
    rewards_all_episodes.append(episode_reward)

### All episodes training completed
After all episodes are finished you now just calculate the average reward per thousand episodes from your list that contains the rewards for all episodes so that you can print it out and see how the rewards changed over time.

In [ ]:
# Average binary episode rewards in blocks to obtain an understandable success rate.
# A larger block shows the long-term learning trend more clearly.
block_size = 1_000
rewards_array = np.asarray(rewards_all_episodes)
for start in range(0, num_episodes, block_size):
    block = rewards_array[start:start + block_size]
    print(f"Episodes {start + 1:5d}-{start + len(block):5d}: average reward = {block.mean():.3f}")

### Interpretation

Because Frozen Lake gives reward `1` only at the goal, the average reward in a block is also the empirical success rate for that block. The exact values depend on the seed and hyperparameters, so interpret the values printed by your own run rather than fixed percentages. Compare learning curves across your planned experiments and keep all other settings constant when testing one parameter.

Finally, inspect the learned Q-table. Each row represents a state and each column an action; larger entries indicate actions with higher estimated discounted return.

In [ ]:
# The final table encodes the learned policy; close the environment after use.
print("Learned Q-table:")
print(q_table)
env.close()